# ALDC vs Baseline: Análisis Completo / Full Comparison

Comparación de 3 escenarios × 2 agentes × 2 modelos sobre 3 instancias de bug-fix BCApps.

| Escenario | Descripción |
|-----------|-------------|
| **Baseline** | Claude/Copilot vanilla + altool MCP, sin custom instructions |
| **ALDC + al-developer-bench** | ALDC completo (instructions + 11 skills + 8 rules) con agente implementador directo |
| **ALDC + al-conductor-bench** | ALDC completo con orquestador TDD multi-agente (planning → implement → review) |

**Caveat**: ALDC está diseñado para uso interactivo con HITL (Human-in-the-Loop) gates. En BC-Bench se testea solo la parte ejecutora en modo autónomo ("bench mode"), sin validaciones humanas. El consumo de tokens es inherentemente mayor (~300KB de contexto ALDC).

**n=3 instancias** — análisis exploratorio, sin significancia estadística.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sys.path.insert(0, str(Path.cwd().parent))

RESULT_DIR = Path.cwd().parent / "result" / "bug-fix"
INSTANCES = ["microsoft__BCApps-5633", "microsoft__BCApps-4822", "microsoft__BCApps-4699"]
INSTANCE_LABELS = {"microsoft__BCApps-5633": "BCApps-5633", "microsoft__BCApps-4822": "BCApps-4822", "microsoft__BCApps-4699": "BCApps-4699"}
INSTANCE_DIFFICULTY = {"microsoft__BCApps-5633": "Hard", "microsoft__BCApps-4822": "Medium", "microsoft__BCApps-4699": "Easy"}

# Map (agent, model, scenario) -> result folder name
SETUP_MAP = {
    ("Claude Code", "sonnet-4-6", "Baseline"): "claude-baseline-sonnet-4-6",
    ("Claude Code", "sonnet-4-6", "ALDC+developer"): "claude-aldc-al-developer-bench-sonnet-4-6",
    ("Claude Code", "sonnet-4-6", "ALDC+conductor"): "claude-aldc-al-conductor-bench-sonnet-4-6",
    ("Claude Code", "opus-4-6", "Baseline"): "claude-baseline-opus-4-6",
    ("Claude Code", "opus-4-6", "ALDC+developer"): "claude-aldc-al-developer-bench-opus-4-6",
    ("Claude Code", "opus-4-6", "ALDC+conductor"): "claude-aldc-al-conductor-bench-opus-4-6",
    ("Copilot", "sonnet-4-6", "Baseline"): "copilot-baseline-sonnet-4-6",
    ("Copilot", "sonnet-4-6", "ALDC+developer"): "copilot-aldc-al-developer-bench-sonnet-4-6",
    ("Copilot", "sonnet-4-6", "ALDC+conductor"): "copilot-aldc-al-conductor-bench-sonnet-4-6",
    ("Copilot", "opus-4-6", "Baseline"): "copilot-baseline-opus-4-6",
    ("Copilot", "opus-4-6", "ALDC+developer"): "copilot-aldc-al-developer-bench-opus-4-6",
    ("Copilot", "opus-4-6", "ALDC+conductor"): "copilot-aldc-al-conductor-bench-opus-4-6",
}


def load_aldc_results() -> pd.DataFrame:
    """Load all ALDC comparison results into a unified DataFrame."""
    rows = []
    for (agent, model, scenario), folder_name in SETUP_MAP.items():
        folder = RESULT_DIR / folder_name
        if not folder.exists():
            continue
        for instance_id in INSTANCES:
            jsonl_file = folder / f"{instance_id}.jsonl"
            if not jsonl_file.exists():
                continue
            for run_idx, line in enumerate(jsonl_file.read_text(encoding="utf-8").splitlines()):
                if not line.strip():
                    continue
                data = json.loads(line)
                m = data.get("metrics", {})
                exp = data.get("experiment", {})
                au = m.get("aldc_usage", {})
                ev = exp.get("aldc_evidence", {})
                rows.append({
                    "agent": agent,
                    "model": model,
                    "scenario": scenario,
                    "instance_id": instance_id,
                    "instance": INSTANCE_LABELS.get(instance_id, instance_id),
                    "difficulty": INSTANCE_DIFFICULTY.get(instance_id, "?"),
                    "run_idx": run_idx,
                    "resolved": data.get("resolved", False),
                    "build": data.get("build", False),
                    "turns": m.get("turn_count"),
                    "time_s": m.get("execution_time"),
                    "prompt_tokens": m.get("prompt_tokens"),
                    "completion_tokens": m.get("completion_tokens"),
                    "tool_usage": m.get("tool_usage"),
                    "agent_confirmed": au.get("custom_agent_confirmed"),
                    "skills_invoked": au.get("skills_invoked", {}),
                    "subagents_invoked": au.get("subagents_invoked", {}),
                    "rules_inlined": ev.get("rules_inlined"),
                    "rules_count": ev.get("rules_inlined_count"),
                    "aldc_files_count": len(ev.get("files", [])),
                    "custom_agent": exp.get("custom_agent"),
                    "custom_instructions": exp.get("custom_instructions"),
                })
    return pd.DataFrame(rows)


df = load_aldc_results()
print(f"Loaded {len(df)} result rows across {df['instance_id'].nunique()} instances")
print(f"Setups with data: {df.groupby(['agent', 'model', 'scenario']).ngroups}")
print(f"\nCoverage matrix (rows with data):")
print(df.groupby(["instance", "agent", "model", "scenario"])["resolved"].count().unstack("scenario").fillna(0).astype(int))

Loaded 25 result rows across 3 instances
Setups with data: 12

Coverage matrix (rows with data):
scenario                            ALDC+conductor  ALDC+developer  Baseline
instance    agent       model                                               
BCApps-4699 Claude Code opus-4-6                 0               1         0
                        sonnet-4-6               1               0         1
            Copilot     opus-4-6                 1               0         1
                        sonnet-4-6               0               1         0
BCApps-4822 Claude Code opus-4-6                 1               1         1
                        sonnet-4-6               1               1         1
            Copilot     opus-4-6                 1               1         0
                        sonnet-4-6               1               1         1
BCApps-5633 Claude Code opus-4-6                 0               0         2
                        sonnet-4-6               1      

## 1. Heatmap de Resolución / Resolution Heatmap

Filas = instancia × modelo. Columnas = escenario × agente.
- Verde = resolved. Rojo = failed. Gris = no testado. Amarillo = parcial (e.g. 1/2 runs).

In [2]:
# Build heatmap matrix
# Rows: instance × model (ordered by difficulty)
# Columns: scenario × agent

SCENARIOS = ["Baseline", "ALDC+developer", "ALDC+conductor"]
AGENTS = ["Claude Code", "Copilot"]
MODELS = ["sonnet-4-6", "opus-4-6"]
INST_ORDER = ["microsoft__BCApps-5633", "microsoft__BCApps-4822", "microsoft__BCApps-4699"]

row_labels = []
col_labels = []
z_matrix = []
text_matrix = []

# Build column labels
for scenario in SCENARIOS:
    for agent in AGENTS:
        col_labels.append(f"{scenario}<br><sub>{agent}</sub>")

# Build rows: instance × model
for inst_id in INST_ORDER:
    inst_label = INSTANCE_LABELS[inst_id]
    diff = INSTANCE_DIFFICULTY[inst_id]
    for model in MODELS:
        row_labels.append(f"{inst_label} ({diff})<br><sub>{model}</sub>")
        row_z = []
        row_text = []
        for scenario in SCENARIOS:
            for agent in AGENTS:
                subset = df[(df["instance_id"] == inst_id) & (df["model"] == model) &
                            (df["scenario"] == scenario) & (df["agent"] == agent)]
                if subset.empty:
                    row_z.append(-1)  # not tested
                    row_text.append("—")
                else:
                    n_runs = len(subset)
                    n_resolved = subset["resolved"].sum()
                    avg_tokens = subset["prompt_tokens"].mean()
                    if n_runs == 1:
                        if n_resolved == 1:
                            row_z.append(1.0)
                            row_text.append(f"✅<br>{avg_tokens/1000:.0f}K")
                        else:
                            row_z.append(0.0)
                            row_text.append(f"❌<br>{avg_tokens/1000:.0f}K")
                    else:
                        ratio = n_resolved / n_runs
                        row_z.append(ratio)
                        row_text.append(f"{n_resolved}/{n_runs}<br>{avg_tokens/1000:.0f}K")
                row_z_val = row_z[-1]
        z_matrix.append(row_z)
        text_matrix.append(row_text)

# Custom colorscale: -1=gray, 0=red, 0.5=yellow, 1=green
# Normalize -1..1 to 0..1 for plotly
z_norm = [[(v + 1) / 2 for v in row] for row in z_matrix]

colorscale = [
    [0.0, "#bdc3c7"],   # -1 = not tested (gray)
    [0.25, "#e74c3c"],  # 0 = all failed (red)
    [0.5, "#e74c3c"],   # just below midpoint
    [0.625, "#f39c12"], # 0.5 = partial (yellow)
    [0.75, "#f39c12"],  # partial
    [1.0, "#2ecc71"],   # 1 = all passed (green)
]

fig = go.Figure(data=go.Heatmap(
    z=z_norm,
    x=col_labels,
    y=row_labels,
    text=text_matrix,
    texttemplate="%{text}",
    textfont={"size": 12},
    colorscale=colorscale,
    showscale=False,
    hovertemplate="<b>%{y}</b><br>%{x}<br>%{text}<extra></extra>",
))

fig.update_layout(
    title="Resolution Heatmap: ALDC vs Baseline (3 instances × 2 models × 6 setups)",
    height=500,
    width=1000,
    yaxis=dict(autorange="reversed", tickfont=dict(size=11)),
    xaxis=dict(tickfont=dict(size=10), side="top"),
    margin=dict(l=200, r=20, t=80, b=20),
)

fig.show()

## 2. Tabla Resumen / Summary Table

Todas las celdas con datos disponibles. Para multi-run, se muestra la primera ejecución.

In [3]:
# Summary table — one row per unique (instance, agent, model, scenario), take run_idx=0
summary = df[df["run_idx"] == 0].copy()
summary["Resolved"] = summary["resolved"].map({True: "✅", False: "❌"})
summary["Build"] = summary["build"].map({True: "✅", False: "❌"})
summary["Tokens (K)"] = (summary["prompt_tokens"] / 1000).round(0).astype(int)
summary["Time (s)"] = summary["time_s"].round(0).astype(int)

display_cols = ["instance", "agent", "model", "scenario", "Resolved", "Build", "turns", "Time (s)", "Tokens (K)"]
summary_display = summary[display_cols].sort_values(
    ["instance", "model", "scenario", "agent"],
    key=lambda x: x.map({"microsoft__BCApps-5633": 0, "microsoft__BCApps-4822": 1, "microsoft__BCApps-4699": 2,
                          "Baseline": 0, "ALDC+developer": 1, "ALDC+conductor": 2}.get) if x.name in ["instance", "scenario"] else x
).reset_index(drop=True)

summary_display.columns = ["Instance", "Agent", "Model", "Scenario", "Resolved", "Build", "Turns", "Time (s)", "Tokens (K)"]
summary_display.style.set_properties(**{"text-align": "center"}).set_table_styles(
    [{"selector": "th", "props": [("text-align", "center")]}]
)

,Instance,Agent,Model,Scenario,Resolved,Build,Turns,Time (s),Tokens (K)
0,BCApps-5633,Claude Code,opus-4-6,Baseline,❌,✅,13,366,327
1,BCApps-4822,Claude Code,opus-4-6,Baseline,✅,✅,11,119,182
2,BCApps-4699,Copilot,opus-4-6,Baseline,✅,✅,15,120,412
3,BCApps-4822,Claude Code,opus-4-6,ALDC+developer,✅,✅,17,127,689
4,BCApps-4699,Claude Code,opus-4-6,ALDC+developer,✅,✅,21,109,785
5,BCApps-4822,Copilot,opus-4-6,ALDC+developer,✅,✅,22,260,1300
6,BCApps-4822,Claude Code,opus-4-6,ALDC+conductor,✅,✅,18,106,776
7,BCApps-4822,Copilot,opus-4-6,ALDC+conductor,✅,✅,21,231,1200
8,BCApps-4699,Copilot,opus-4-6,ALDC+conductor,✅,✅,22,141,1100
9,BCApps-5633,Claude Code,sonnet-4-6,Baseline,❌,✅,26,460,1140


## 3. Token Overhead / Coste de Contexto ALDC

ALDC carga ~300KB de instrucciones, skills y reglas. El overhead de tokens es estructural.
La pregunta: ese contexto adicional, ¿mejora la resolución o solo infla el coste?

In [4]:
# Token overhead analysis — bar chart per instance, colored by resolved
first_runs = df[df["run_idx"] == 0].copy()

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[f"{INSTANCE_LABELS[i]} ({INSTANCE_DIFFICULTY[i]})" for i in INST_ORDER],
    shared_yaxes=True,
)

color_map = {True: "#2ecc71", False: "#e74c3c"}

for col_idx, inst_id in enumerate(INST_ORDER, 1):
    inst_data = first_runs[first_runs["instance_id"] == inst_id].copy()
    inst_data["label"] = inst_data["scenario"] + "<br>" + inst_data["agent"] + "<br><sub>" + inst_data["model"] + "</sub>"
    inst_data = inst_data.sort_values(["scenario", "agent", "model"])

    fig.add_trace(
        go.Bar(
            x=inst_data["label"],
            y=inst_data["prompt_tokens"] / 1000,
            marker_color=[color_map[r] for r in inst_data["resolved"]],
            text=[f"{'✅' if r else '❌'} {t/1000:.0f}K" for r, t in zip(inst_data["resolved"], inst_data["prompt_tokens"])],
            textposition="outside",
            textfont=dict(size=9),
            showlegend=False,
        ),
        row=1, col=col_idx,
    )

fig.update_layout(
    title="Prompt Tokens por configuración (verde=resolved, rojo=failed)",
    height=500,
    width=1200,
    yaxis_title="Prompt tokens (K)",
)
fig.show()

# Compute overhead multipliers where baseline exists for comparison
print("\n=== Token Overhead Multiplier (ALDC / Baseline) ===")
print("(Only for cells where both baseline and ALDC exist for same agent+model+instance)\n")

for inst_id in INST_ORDER:
    inst_label = INSTANCE_LABELS[inst_id]
    for agent in AGENTS:
        for model in MODELS:
            baseline = first_runs[(first_runs["instance_id"] == inst_id) & (first_runs["agent"] == agent) &
                                  (first_runs["model"] == model) & (first_runs["scenario"] == "Baseline")]
            if baseline.empty:
                continue
            base_tokens = baseline.iloc[0]["prompt_tokens"]
            base_resolved = baseline.iloc[0]["resolved"]
            for scenario in ["ALDC+developer", "ALDC+conductor"]:
                aldc = first_runs[(first_runs["instance_id"] == inst_id) & (first_runs["agent"] == agent) &
                                  (first_runs["model"] == model) & (first_runs["scenario"] == scenario)]
                if aldc.empty:
                    continue
                aldc_tokens = aldc.iloc[0]["prompt_tokens"]
                aldc_resolved = aldc.iloc[0]["resolved"]
                ratio = aldc_tokens / base_tokens if base_tokens > 0 else float("inf")
                res_change = "same" if base_resolved == aldc_resolved else ("improved" if aldc_resolved else "regressed")
                print(f"  {inst_label} | {agent} {model} | {scenario}: {ratio:.1f}x tokens | resolution: {res_change}")


=== Token Overhead Multiplier (ALDC / Baseline) ===
(Only for cells where both baseline and ALDC exist for same agent+model+instance)

  BCApps-5633 | Claude Code sonnet-4-6 | ALDC+developer: 1.0x tokens | resolution: same
  BCApps-5633 | Claude Code sonnet-4-6 | ALDC+conductor: 2.8x tokens | resolution: same
  BCApps-5633 | Copilot sonnet-4-6 | ALDC+developer: 3.3x tokens | resolution: improved
  BCApps-5633 | Copilot sonnet-4-6 | ALDC+conductor: 5.5x tokens | resolution: same
  BCApps-4822 | Claude Code sonnet-4-6 | ALDC+developer: 3.0x tokens | resolution: same
  BCApps-4822 | Claude Code sonnet-4-6 | ALDC+conductor: 3.8x tokens | resolution: same
  BCApps-4822 | Claude Code opus-4-6 | ALDC+developer: 3.8x tokens | resolution: same
  BCApps-4822 | Claude Code opus-4-6 | ALDC+conductor: 4.3x tokens | resolution: same
  BCApps-4822 | Copilot sonnet-4-6 | ALDC+developer: 1.5x tokens | resolution: regressed
  BCApps-4822 | Copilot sonnet-4-6 | ALDC+conductor: 1.1x tokens | resolution: 

## 4. Eficiencia: Tiempo y Turns / Efficiency: Time and Turns

In [5]:
# Time and turns comparison — grouped bar chart
first_runs = df[df["run_idx"] == 0].copy()
first_runs["config"] = first_runs["agent"] + " " + first_runs["model"]

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Turns (fewer = more efficient)", "Execution Time (s)"],
)

scenario_colors = {"Baseline": "#3498db", "ALDC+developer": "#e67e22", "ALDC+conductor": "#9b59b6"}

for col_idx, (metric, ylabel) in enumerate([("turns", "Turns"), ("time_s", "Time (s)")], 1):
    for scenario in SCENARIOS:
        subset = first_runs[first_runs["scenario"] == scenario].sort_values(["instance_id", "agent", "model"])
        labels = [f"{r['instance']}<br>{r['agent']}<br><sub>{r['model']}</sub>" for _, r in subset.iterrows()]
        markers = ["✅" if r else "❌" for r in subset["resolved"]]
        fig.add_trace(
            go.Bar(
                x=labels,
                y=subset[metric],
                name=scenario,
                marker_color=scenario_colors[scenario],
                text=[f"{m} {v:.0f}" for m, v in zip(markers, subset[metric])],
                textposition="outside",
                textfont=dict(size=8),
                showlegend=(col_idx == 1),
            ),
            row=1, col=col_idx,
        )

fig.update_layout(
    title="Eficiencia por configuración (✅=resolved, ❌=failed)",
    height=500,
    width=1200,
    barmode="group",
    legend=dict(orientation="h", yanchor="bottom", y=1.08),
)
fig.show()

## 5. El Problema de la Ceguera / The Blindness Problem

ALDC despliega 32 archivos (~300KB) en el directorio `.claude/` del repositorio. Pero no podemos
observar directamente si el modelo los lee, los procesa, o los ignora. Solo tenemos evidencia indirecta:

- `custom_agent_confirmed`: ¿apareció el nombre del agente en los logs?
- `skills_invoked`: ¿se invocó alguna skill explícitamente via tool?
- `subagents_invoked`: ¿se delegó a subagentes (planning, implement, review)?
- `rules_inlined`: ¿se inyectaron las 8 coding rules en el CLAUDE.md?

**Hallazgo clave**: Claude Code nunca invoca subagentes. Copilot siempre invoca los 3, incluso
cuando el agente es `al-developer-bench` (que no debería orquestar subagentes).

In [6]:
# ALDC Evidence verification table
aldc_runs = df[(df["scenario"] != "Baseline") & (df["run_idx"] == 0)].copy()

evidence_rows = []
for _, row in aldc_runs.iterrows():
    skills = row["skills_invoked"]
    subagents = row["subagents_invoked"]
    evidence_rows.append({
        "Instance": row["instance"],
        "Agent": row["agent"],
        "Model": row["model"],
        "Scenario": row["scenario"],
        "Resolved": "✅" if row["resolved"] else "❌",
        "Agent Confirmed": "✅" if row["agent_confirmed"] else "❌",
        "Skills Invoked": len(skills) if skills else 0,
        "Subagents Invoked": ", ".join(subagents.keys()) if subagents else "—",
        "Rules Inlined": f"{row['rules_count']}/8" if row["rules_count"] else "—",
        "ALDC Files": row["aldc_files_count"],
    })

evidence_df = pd.DataFrame(evidence_rows)
evidence_df.style.set_properties(**{"text-align": "center"}).set_table_styles(
    [{"selector": "th", "props": [("text-align", "center")]}]
)

,Instance,Agent,Model,Scenario,Resolved,Agent Confirmed,Skills Invoked,Subagents Invoked,Rules Inlined,ALDC Files
0,BCApps-5633,Claude Code,sonnet-4-6,ALDC+developer,❌,✅,0,—,8/8,32
1,BCApps-4822,Claude Code,sonnet-4-6,ALDC+developer,✅,✅,0,—,8/8,32
2,BCApps-5633,Claude Code,sonnet-4-6,ALDC+conductor,❌,✅,0,—,8/8,32
3,BCApps-4822,Claude Code,sonnet-4-6,ALDC+conductor,✅,✅,0,—,8/8,32
4,BCApps-4699,Claude Code,sonnet-4-6,ALDC+conductor,✅,✅,0,—,8/8,32
5,BCApps-4822,Claude Code,opus-4-6,ALDC+developer,✅,✅,0,—,8/8,32
6,BCApps-4699,Claude Code,opus-4-6,ALDC+developer,✅,✅,0,—,8/8,32
7,BCApps-4822,Claude Code,opus-4-6,ALDC+conductor,✅,✅,0,—,8/8,32
8,BCApps-5633,Copilot,sonnet-4-6,ALDC+developer,✅,✅,0,"al-planning-subagent, al-implement-subagent, al-review-subagent",8/8,32
9,BCApps-4822,Copilot,sonnet-4-6,ALDC+developer,❌,✅,0,"al-planning-subagent, al-implement-subagent, al-review-subagent",8/8,32


In [7]:
# Visualize the Claude vs Copilot subagent asymmetry
evidence_summary = aldc_runs.copy()
evidence_summary["has_subagents"] = evidence_summary["subagents_invoked"].apply(lambda x: len(x) > 0 if x else False)

pivot = evidence_summary.groupby(["agent", "scenario"])["has_subagents"].mean() * 100
print("=== Subagent Invocation Rate (% of runs) ===\n")
print(pivot.unstack().to_string())
print("\nClaude Code: 0% subagent invocation across ALL ALDC runs")
print("Copilot: 100% subagent invocation across ALL ALDC runs (even with developer agent)")
print("\nThis asymmetry suggests fundamentally different execution paths between the two runners.")

=== Subagent Invocation Rate (% of runs) ===

scenario     ALDC+conductor  ALDC+developer
agent                                      
Claude Code             0.0             0.0
Copilot               100.0           100.0

Claude Code: 0% subagent invocation across ALL ALDC runs
Copilot: 100% subagent invocation across ALL ALDC runs (even with developer agent)

This asymmetry suggests fundamentally different execution paths between the two runners.


## 6. Comparación de Tool Usage / Tool Usage Comparison

¿Las instrucciones ALDC causan más exploración (Read, Grep, Glob) y menos edición directa?
Datos solo disponibles para Claude Code (Copilot no reporta tool usage granular por herramienta).

In [8]:
# Tool usage comparison (Claude Code only — has granular tool_usage data)
claude_runs = df[(df["agent"] == "Claude Code") & (df["run_idx"] == 0) & (df["tool_usage"].apply(lambda x: bool(x)))].copy()

tool_rows = []
for _, row in claude_runs.iterrows():
    tu = row["tool_usage"]
    if not tu:
        continue
    tool_rows.append({
        "Instance": row["instance"],
        "Model": row["model"],
        "Scenario": row["scenario"],
        "Resolved": "✅" if row["resolved"] else "❌",
        "Read": tu.get("Read", 0),
        "Grep": tu.get("Grep", 0),
        "Glob": tu.get("Glob", 0),
        "Edit": tu.get("Edit", 0),
        "Bash": tu.get("Bash", 0),
        "Agent": tu.get("Agent", 0),
        "TodoWrite": tu.get("TodoWrite", 0),
        "ToolSearch": tu.get("ToolSearch", 0),
        "Exploration (R+G+G)": tu.get("Read", 0) + tu.get("Grep", 0) + tu.get("Glob", 0),
    })

tool_df = pd.DataFrame(tool_rows)
if not tool_df.empty:
    # Show exploration vs edit ratio
    print("=== Tool Usage: Exploration (Read+Grep+Glob) vs Edit ===\n")
    for scenario in SCENARIOS:
        s = tool_df[tool_df["Scenario"] == scenario]
        if s.empty:
            continue
        avg_explore = s["Exploration (R+G+G)"].mean()
        avg_edit = s["Edit"].mean()
        ratio = avg_explore / avg_edit if avg_edit > 0 else float("inf")
        print(f"  {scenario}: explore={avg_explore:.0f} vs edit={avg_edit:.0f} (ratio {ratio:.1f}x)")

    print()
    display(tool_df.style.set_properties(**{"text-align": "center"}))

=== Tool Usage: Exploration (Read+Grep+Glob) vs Edit ===

  Baseline: explore=66 vs edit=6 (ratio 11.9x)
  ALDC+developer: explore=32 vs edit=4 (ratio 8.7x)
  ALDC+conductor: explore=35 vs edit=6 (ratio 5.8x)



,Instance,Model,Scenario,Resolved,Read,Grep,Glob,Edit,Bash,Agent,TodoWrite,ToolSearch,Exploration (R+G+G)
0,BCApps-5633,sonnet-4-6,Baseline,❌,30,16,7,6,2,1,3,1,53
1,BCApps-4822,sonnet-4-6,Baseline,✅,37,17,8,9,4,1,3,1,62
2,BCApps-4699,sonnet-4-6,Baseline,✅,4,5,1,3,0,0,0,0,10
3,BCApps-5633,sonnet-4-6,ALDC+developer,❌,29,0,9,1,8,1,0,0,38
4,BCApps-4822,sonnet-4-6,ALDC+developer,✅,46,0,12,2,10,1,0,0,58
5,BCApps-5633,sonnet-4-6,ALDC+conductor,❌,25,13,8,7,4,1,5,1,46
6,BCApps-4822,sonnet-4-6,ALDC+conductor,✅,38,28,9,11,5,1,5,1,75
7,BCApps-4699,sonnet-4-6,ALDC+conductor,✅,3,2,0,3,0,0,0,0,5
8,BCApps-5633,opus-4-6,Baseline,❌,47,8,10,2,18,2,0,0,65
9,BCApps-4822,opus-4-6,Baseline,✅,96,31,15,8,26,4,0,0,142


## 7. Conclusiones / Conclusions

### 7.1 Resolución: ALDC no mejora la tasa de bug-fix / Resolution: ALDC does not improve bug-fix rate

Con n=3 instancias, **ningún escenario ALDC supera consistentemente al baseline**:
- BCApps-5633 (Hard): todos fallan con sonnet. Opus baseline pasa 1/2 runs. Solo Copilot+developer resuelve (73 turns, 1.2M tokens).
- BCApps-4822 (Medium): todos pasan excepto Copilot+developer sonnet (regresión ALDC-inducida).
- BCApps-4699 (Easy): todos pasan.

**El ALDC no añade capacidad de resolución en bug-fix**. En el caso más interesante (BCApps-5633), la resolución viene del modelo (opus) o de la persistencia extrema (Copilot, 73 turns), no de las instrucciones ALDC.

### 7.2 Tokens: 3-4x de overhead sistemático / Tokens: 3-4x systematic overhead

El overhead medio de ALDC es **3.0-4.3x** sobre baseline (developer y conductor respectivamente). Este coste es estructural: ~300KB de instrucciones + skills + reglas se cargan al principio y persisten en contexto.

Dato revelador: en BCApps-5633, ALDC+developer usa **menos tokens que baseline** (0.97x) — porque falla más rápido y de manera diferente, no porque sea más eficiente.

### 7.3 El "problema de la ceguera" / The "blindness problem"

- **Skills**: 0 invocaciones explícitas en 16 runs ALDC. Las skills existen como archivos `.md` pero nunca se invocan via tool. El modelo puede leerlas via Read/Glob pero no podemos verificarlo.
- **Subagentes**: asimetría total entre runners. Claude Code nunca delega a subagentes (ni siquiera con conductor). Copilot siempre delega los 3 (incluso con developer). Esto sugiere que el routing de agentes funciona fundamentalmente diferente entre los dos runners.
- **Agent confirmed**: True en todos los runs ALDC, confirmando que al menos el nombre del agente se reconoce.

### 7.4 ¿Es válido el enfoque executor-only? / Is the executor-only approach valid?

**Sí, parcialmente**. BC-Bench demuestra que:
1. El ejecutor ALDC **puede funcionar autónomamente** (agent_confirmed=true, builds pasan, genera patches razonables).
2. El overhead de contexto **no se traduce en mejor resolución** para bug-fix.
3. Las validation gates HITL del ALDC interactivo no son solo ceremonia — son donde el valor se materializa (corrección de dirección, foco en el lugar correcto del fix).

**No es justo concluir que ALDC "no funciona"** a partir de estos datos. Es como evaluar un piloto de F1 conduciéndolo por una calle residencial: el contexto no pone a prueba las capacidades diseñadas. ALDC está diseñado para:
- Desarrollo guiado por spec con validación humana
- TDD completo (RED → GREEN → REFACTOR) con gates
- Features medianas/grandes donde la orquestación multi-agente justifica su coste

Bug-fix de ~10 líneas en un monorepo es el **peor caso posible** para ALDC: máximo overhead, mínimo beneficio.

### 7.5 El modelo importa más que las instrucciones / Model matters more than instructions

Opus-4-6 baseline resuelve BCApps-5633 (1/2 runs) donde **ningún** sonnet-4-6 (baseline o ALDC) lo logra. Un upgrade de modelo entrega lo que 300KB de instrucciones no pueden. Esto es consistente con la literatura de SWE-Bench: la capacidad base del modelo es el predictor dominante.

### 7.6 Recomendaciones / Recommendations

1. **Para ALDC en benchmark**: Crear un modo "pruned" con solo 3-4 skills relevantes al tipo de tarea (no las 11 completas). Reducir el contexto inicial de ~300KB a ~50KB.
2. **Para BC-Bench**: Añadir un baseline "context-aware" (solo CLAUDE.md + reglas, sin skills ni agentes) para aislar el efecto de las instrucciones del overhead de skills.
3. **Para evaluación futura**: Test-generation y feature-development son las categorías naturales de ALDC conductor. Bug-fix es la peor para medirlo.
4. **Ablation study**: CLAUDE.md solo → +rules → +skills → full ALDC, para identificar qué capa aporta y cuál resta.

### 7.7 Limitaciones / Limitations

| Limitación | Impacto |
|-----------|---------|
| n=3 instancias | Sin significancia estadística posible |
| Single-run mayoritario | Sin estimación de varianza |
| Matriz sparse | No todos los combos testados |
| Solo bug-fix | Conductor diseñado para TDD/test-gen |
| "Blindness" | No sabemos si el modelo realmente procesa las instrucciones |
| Bench mode | Sin HITL gates — no es ALDC completo |